# ENEMDU 2021-2025 — Pipeline Medallón (Bronze → Silver → Gold)

**Proyecto:** UCUENCA-SABE · MVP Dashboard de Empleabilidad
**Fuente:** Encuesta Nacional de Empleo, Desempleo y Subempleo (ENEMDU) — INEC, microdatos anuales de personas.

## Objetivo del caso de uso
1. Analizar la **empleabilidad de graduados universitarios** (2021-2025).
2. Identificar **sobrecalificación**: graduados ocupados en ocupaciones que no requieren título superior (CIUO-08, grandes grupos 4-9).
3. Generar **agregados listos para Power BI**.

## Estructura del notebook
| Sección | Contenido | Módulo |
|---|---|---|
| 1 | Ingesta Bronze (Kaggle) | `src/ingestion/kaggle_downloader.py` |
| 2 | **Objetivo 1** — Auditoría y análisis de esquemas | `enemdu_schema_analyzer.py` |
| 3 | **Objetivo 2** — Construcción de Silver | `enemdu_silver_builder.py` |
| 4 | **Objetivo 3** — Métricas de empleabilidad (Gold) | `enemdu_gold_builder.py` |
| 5 | **Objetivo 4** — Validación, visualizaciones y export | notebook |

> **Nota:** la capa Bronze está reducida a modo de ejemplo; el pipeline está diseñado para el volumen completo (lectura por `usecols`, tipado explícito y Parquet comprimido).

---
## 1. Ingesta — Capa Bronze
Descarga desde Kaggle y clasificación en `diccionarios/` y `microdatos_csv/`. Bronze es **inmutable**: sirve de respaldo y trazabilidad; ninguna transformación se escribe aquí.

Si ya se descargó ya no correr la kaggle sino toma 9min aprox

In [ ]:
import sys
from pathlib import Path

# Inyectar la raíz del proyecto al sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG, get_external_bronze_dir
from src.ingestion.kaggle_downloader import download_and_organize_enemdu

print(f"✅ Proyecto cargado correctamente: {CONFIG['project']['name']}")
print(f"📁 Ruta raíz: {PROJECT_ROOT}")
print(f"📊 Sources disponibles: {list(CONFIG['sources'].keys())}")

In [ ]:
# Ahora descarga
print("🚀 Iniciando descarga desde Kaggle...")
try:
    inventario = download_and_organize_enemdu()
    print("✅ ¡Proceso completado!")
except Exception as e:
    print(f"❌ Error: {e}")

### 1.1 Chequeo de años nuevos
Kaggle no expone una API de "cambios"; la estrategia es comparar los años presentes en local contra el año esperado y volver a descargar sólo si falta alguno.

In [ ]:
from pathlib import Path

def check_nuevos_anios(anio_esperado: int = 2025, descargar: bool = False) -> list[int]:
    """Devuelve los años ausentes en Bronze y, opcionalmente, relanza la ingesta."""
    csv_dir = PROJECT_ROOT / "data" / "bronze" / "externas" / "enemdu" / "microdatos_csv"
    anios_locales = sorted(
        int(m.group()) for f in csv_dir.glob("*.csv")
        if (m := __import__("re").search(r"(?:19|20)\d{2}", f.stem))
    )
    faltantes = [a for a in range(2021, anio_esperado + 1) if a not in anios_locales]

    print(f"📂 Años locales: {anios_locales}")
    if faltantes:
        print(f"📥 Faltan años: {faltantes}")
        if descargar:
            download_and_organize_enemdu()
    else:
        print("✅ Ya tienes los datos más recientes")
    return faltantes

check_nuevos_anios(2025)

In [ ]:
print(CONFIG["sources"].keys())

---
## 2. Objetivo 1 — Auditoría y análisis de esquemas

**Pregunta que responde esta sección:** ¿qué contiene realmente Bronze y qué se puede llevar a Silver sin romper la comparabilidad 2021-2025?

`ENEMDUSchemaAnalyzer` hace cuatro cosas:
1. Lee **todos los diccionarios XLSX** (detectando automáticamente la fila de encabezado, que cambia entre años).
2. Lee **sólo la cabecera** de cada CSV (detectando encoding, separador y decimal — el INEC alterna `;`/`,` y UTF-8/Latin-1).
3. Cruza *esquema declarado* (diccionario) vs. *esquema real* (CSV).
4. Audita calidad: nulos, duplicados de llave, rangos y presencia de variables críticas.

In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src.processing import ENEMDUSchemaAnalyzer, ENEMDUSilverBuilder, ENEMDUGoldBuilder
from src.processing.io_utils import ENEMDUPaths

paths = ENEMDUPaths.build(PROJECT_ROOT).ensure()
for k, v in paths.as_dict().items():
    print(f"{k:>14}: {v}")

### 2.1 Inventario de la capa Bronze

In [ ]:
analyzer = ENEMDUSchemaAnalyzer(paths=paths, years=range(2021, 2026))
inventario_bronze = analyzer.inventory()
display(inventario_bronze)

print(f"\nTotal archivos: {len(inventario_bronze)} | "
      f"Peso Bronze: {inventario_bronze['peso_mb'].sum():.1f} MB")

### 2.2 Análisis de esquemas por año

In [ ]:
schemas = analyzer.analyze_all()
resumen = analyzer.summary()
display(resumen)

print("Lectura:")
print("  · cobertura_% = columnas del CSV documentadas en el diccionario")
print("  · solo_en_dicc = variables documentadas que NO llegaron al CSV (revisar entrega del INEC)")
print("  · cols_exclusivas = columnas que existen únicamente en ese año")

### 2.3 Columnas comunes vs. específicas por año

In [ ]:
comunes = analyzer.common_columns
especificas = analyzer.year_specific

print(f"🔗 Columnas comunes a los {len(schemas)} años: {len(comunes)}")
print(f"🌐 Unión de columnas (todos los años): {len(analyzer.all_columns)}\n")

for anio, cols in especificas.items():
    print(f"  {anio}: {len(cols)} exclusivas → {cols[:12]}{' ...' if len(cols) > 12 else ''}")

matriz = analyzer.presence_matrix()
print("\nDistribución de columnas por estado:")
display(matriz["estado"].value_counts().to_frame("n_columnas"))
display(matriz.head(25))

### 2.4 Resolución de variables críticas
El INEC renombra variables entre publicaciones (`p51a` → `totalhoras`, `ingrl` → `ingreso_laboral`, etc.).
Aquí se resuelve, **año por año**, qué nombre real corresponde a cada variable del modelo de Silver.
Una fila `⚠️ parcial` o `❌ ausente` es una decisión de diseño pendiente, no un error silencioso.

In [ ]:
criticas = analyzer.resolve_critical()
display(criticas)

pendientes = criticas[criticas["estado"] != "✅ completa"]
if len(pendientes):
    print("⚠️ Variables a revisar manualmente en el diccionario:")
    display(pendientes[["variable_silver", "estado"]])
else:
    print("✅ Todas las variables críticas se resolvieron en los 5 años.")

### 2.5 Etiquetas de valores (código → significado)
Los mapeos **no se asumen**: se extraen del diccionario del año. Sólo si no son interpretables se aplica el mapeo *fallback* documentado en `enemdu_mappings.py`, y la procedencia queda registrada en el reporte de calidad.

In [ ]:
for var in ["p10a", "condact", "p02", "area", "rama1"]:
    labels = analyzer.labels_for(var)
    print(f"\n▸ {var} ({len(labels)} categorías)")
    for cod, txt in list(labels.items())[:12]:
        print(f"    {cod:>3} = {txt}")
    if not labels:
        print("    (sin etiquetas en el diccionario → se usará el mapeo fallback)")

### 2.6 Auditoría de calidad de Bronze

In [ ]:
auditoria = analyzer.audit(sample_rows=100_000)
display(auditoria)

ruta_reporte = analyzer.save_report()
print(f"📄 Reporte de esquemas: {ruta_reporte}")

### 2.7 Decisiones de diseño para Silver

A partir de la auditoría se fija el contrato de la capa Silver:

| Decisión | Criterio |
|---|---|
| **Selección de columnas** | No se copia el CSV completo: sólo las variables del modelo de empleabilidad (identificación, geografía, demografía, educación, mercado laboral, ponderador). |
| **Esquema estable** | La unión de variables canónicas se mantiene en todos los años; lo que no exista en un año se completa con `NULL` (NaN), nunca con ceros. |
| **Nombres legibles** | `p02` → `sexo_cod` → `sexo`; `condact` → `condicion_actividad`. Se conserva el código original (`*_cod`) junto a la etiqueta, para trazabilidad y para Power BI. |
| **Códigos de no respuesta** | `999999`, `99`, `-1` en variables continuas → `NaN` (no se imputa en Silver). |
| **Rangos válidos** | edad 0-110, horas 0-126, ingreso 0-100 000, `fexp` ≥ 0. Fuera de rango → `NaN` + registro en el reporte. |
| **Deduplicación** | Sólo si existe identificador de persona real; de lo contrario se advierte y no se borra nada. |
| **Sin imputación ni filtros analíticos** | Filtrar PET (≥15 años) o graduados es responsabilidad de **Gold**, no de Silver. |
| **Formato** | Parquet + Snappy, un archivo unificado (`enemdu_unificado.parquet`). |

---
## 3. Objetivo 2 — Construcción de la capa Silver

`ENEMDUSilverBuilder` aplica, por año: resolución de alias → lectura selectiva → renombrado → esquema estable → tipado y limpieza → decodificación semántica → variables derivadas; y finalmente concatena 2021-2025.

**Derivadas generadas:** `es_pea`, `es_ocupado`, `es_desempleado`, `es_subempleado`, `es_empleo_adecuado`, `es_graduado_superior`, `es_posgrado`, `ciuo_gran_grupo` (+ descripción y nivel de competencia), `ocupacion_requiere_superior`, **`es_sobrecalificado`**, `grupo_edad`, `ingreso_por_hora`, `provincia`.

In [ ]:
# SAMPLE = 50_000 para iterar rápido; None = procesar todo (producción)
SAMPLE = None

silver_builder = ENEMDUSilverBuilder(analyzer=analyzer, paths=paths)
resultado = silver_builder.build(sample_rows=SAMPLE)
silver = resultado.df

print(f"✅ Silver: {silver.shape[0]:,} filas × {silver.shape[1]} columnas")
display(silver.head())

### 3.1 Mapeo aplicado por año (trazabilidad)

In [ ]:
display(resultado.mapping_report.set_index("anio").T)

print("Procedencia de los mapeos de códigos (diccionario | fallback):")
display(pd.DataFrame(resultado.provenance).T)

### 3.2 Validación de consistencia y calidad

In [ ]:
display(resultado.quality)

# Consistencia del tamaño muestral y la población expandida entre años
variacion = resultado.quality.set_index("anio")[["filas", "poblacion_expandida"]].pct_change() * 100
print("Variación interanual (%):")
display(variacion.round(2))

In [ ]:
# Nulos por columna (top 20) y tipos resultantes
nulos = (silver.isna().mean() * 100).sort_values(ascending=False).round(2)
display(nulos.head(20).to_frame("%_nulos"))

print("\nDistribución de la condición de actividad:")
display(silver["condicion_actividad"].value_counts(dropna=False, normalize=True).mul(100).round(2).to_frame("%"))

print("Distribución del nivel de instrucción:")
display(silver["nivel_instruccion"].value_counts(dropna=False, normalize=True).mul(100).round(2).to_frame("%"))

In [ ]:
# Chequeos de integridad: rangos, coherencia lógica y cobertura del ponderador
checks = {
    "edad fuera de rango": int(((silver["edad"] < 0) | (silver["edad"] > 110)).sum()),
    "ingreso negativo": int((silver["ingreso_laboral"] < 0).sum()),
    "ocupado y desempleado a la vez": int((silver["es_ocupado"] & silver["es_desempleado"]).sum()),
    "sobrecalificado sin ser graduado": int((silver["es_sobrecalificado"] & ~silver["es_graduado_superior"]).sum()),
    "sobrecalificado no ocupado": int((silver["es_sobrecalificado"] & ~silver["es_ocupado"]).sum()),
    "filas sin factor de expansión": int(silver["factor_expansion"].isna().sum()),
    "años presentes": silver["anio"].nunique(),
}
display(pd.Series(checks).to_frame("resultado"))
assert checks["ocupado y desempleado a la vez"] == 0, "Inconsistencia en la condición de actividad"
print("✅ Chequeos de integridad superados")

### 3.3 Persistencia de Silver

In [ ]:
ruta_silver = silver_builder.save(resultado, partition_by_year=False)
print(f"💾 {ruta_silver}")
print(f"   Tamaño: {ruta_silver.stat().st_size / 1e6:.2f} MB")

---
## 4. Objetivo 3 — Métricas de empleabilidad (capa Gold)

### Definiciones operativas
| Concepto | Definición |
|---|---|
| **Graduado superior** | Nivel de instrucción *Superior universitaria* o *Post-grado*, con 24 años o más (evita contar a quienes aún cursan). |
| **PET** | Población de 15 años y más. |
| **PEA** | Ocupados + desempleados. |
| **Tasa de desempleo** | Desempleados / PEA. |
| **Tasa de empleo adecuado** | Empleo adecuado-pleno / PEA. |
| **Sobrecalificación** | Graduado ocupado cuyo gran grupo CIUO-08 es 4-9 (niveles de competencia 1-2, no requieren título universitario). |
| **Ponderación** | Todos los indicadores usan el **factor de expansión** (`fexp`); se reporta `n_muestral` y la marca `confiable` (n ≥ 30). |

In [ ]:
gold_builder = ENEMDUGoldBuilder(silver, paths=paths, edad_minima=15, edad_minima_graduado=24)
gold = gold_builder.build()

print("📊 KPI anual — graduados universitarios")
display(gold.kpi_anual[[
    "anio", "n_muestral", "poblacion", "tasa_participacion", "tasa_empleo",
    "tasa_desempleo", "tasa_empleo_adecuado", "tasa_subempleo",
    "tasa_sobrecalificacion", "ingreso_laboral_medio"
]])

### 4.1 Sobrecalificación por gran grupo ocupacional (CIUO-08)

In [ ]:
sobre = gold.sobrecalificacion_ocupacion
display(sobre[sobre["anio"] == sobre["anio"].max()])

resumen_sobre = (sobre.groupby(["anio", "requiere_titulo_superior"])["ocupados"]
                 .sum().unstack().rename(columns={True: "en_ocupación_acorde",
                                                  False: "en_ocupación_no_acorde"}))
resumen_sobre["% sobrecalificación"] = (
    100 * resumen_sobre["en_ocupación_no_acorde"]
    / resumen_sobre.sum(axis=1)).round(2)
display(resumen_sobre)

### 4.2 Territorio, sexo/edad, rama y comparación por nivel educativo

In [ ]:
ultimo = int(silver["anio"].max())

print(f"▸ Top 10 provincias por tasa de desempleo de graduados ({ultimo})")
prov = gold.por_provincia.query("anio == @ultimo and confiable")
display(prov.nlargest(10, "tasa_desempleo")[
    ["provincia", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "tasa_sobrecalificacion", "ingreso_laboral_medio"]])

print(f"▸ Brecha de género ({ultimo})")
display(gold.por_sexo_edad.query("anio == @ultimo")[
    ["sexo", "grupo_edad", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "tasa_sobrecalificacion", "ingreso_laboral_medio", "brecha_desempleo_pp", "confiable"]])

print(f"▸ Ramas de actividad que absorben graduados ({ultimo})")
display(gold.por_rama.query("anio == @ultimo").nlargest(10, "participacion_pct"))

print("▸ Comparativo por nivel de instrucción (todos los niveles)")
display(gold.comparativo_nivel.query("anio == @ultimo")[
    ["nivel_instruccion", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "ingreso_laboral_medio", "tasa_sobrecalificacion"]])

### 4.3 Persistencia de Gold

In [ ]:
salidas = gold_builder.save(gold, export_csv=True)
for nombre, ruta in salidas.items():
    print(f"  {nombre:<32} → {ruta}")

---
## 5. Objetivo 4 — Validación y reporte

### 5.1 Tamaños de muestra y precisión

In [ ]:
muestra = (silver.assign(grad=silver["es_graduado_superior"])
           .groupby("anio")
           .agg(filas_silver=("anio", "size"),
                pet=("es_edad_trabajar", "sum"),
                graduados=("grad", "sum"),
                pea_graduados=("es_pea", "sum"))
           .assign(pct_graduados=lambda d: (100 * d["graduados"] / d["filas_silver"]).round(2)))
display(muestra)

alerta = gold.por_provincia.query("not confiable")
print(f"⚠️ Celdas provincia×año con n < 30 (no publicables): {len(alerta)}")
if len(alerta):
    display(alerta[["anio", "provincia", "n_muestral"]].head(15))

### 5.2 Validación contra cifras oficiales (opcional)
Completa el diccionario con la tasa de desempleo **nacional** publicada por el INEC para cada año
(boletín anual de ENEMDU). La comparación se calcula sobre el total de la PEA, no sólo graduados;
diferencias menores a ~0,5 pp son esperables por redondeo y por el submuestreo de Bronze.

In [ ]:
# Rellenar con los valores del boletín oficial del INEC (dejar None si no se dispone)
TASAS_OFICIALES_INEC = {2021: None, 2022: None, 2023: None, 2024: None, 2025: None}

nacional = ENEMDUGoldBuilder(silver, paths=paths).\
    _rates(ENEMDUGoldBuilder._aggregate(
        silver[silver["edad"].ge(15).fillna(False)], ["anio"]))[["anio", "tasa_desempleo"]]
nacional = nacional.rename(columns={"tasa_desempleo": "tasa_desempleo_calculada"})
nacional["tasa_desempleo_INEC"] = nacional["anio"].map(TASAS_OFICIALES_INEC)
nacional["diferencia_pp"] = (nacional["tasa_desempleo_calculada"]
                             - nacional["tasa_desempleo_INEC"]).round(2)
display(nacional)

if nacional["tasa_desempleo_INEC"].notna().any():
    peor = nacional["diferencia_pp"].abs().max()
    print(f"Máxima diferencia: {peor:.2f} pp", "✅ dentro de lo esperado" if peor < 1 else "⚠️ revisar definiciones")
else:
    print("ℹ️ Sin cifras oficiales cargadas: completa TASAS_OFICIALES_INEC para activar la validación.")

### 5.3 Visualizaciones exploratorias

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9})

kpi = gold.kpi_anual.sort_values("anio")
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0, 0].plot(kpi["anio"], kpi["tasa_desempleo"], marker="o", color="#FF6B35")
axes[0, 0].set_title("Tasa de desempleo — graduados universitarios")
axes[0, 0].set_ylabel("%")

axes[0, 1].plot(kpi["anio"], kpi["tasa_empleo_adecuado"], marker="o", color="#1B998B")
axes[0, 1].plot(kpi["anio"], kpi["tasa_subempleo"], marker="s", color="#C7522A")
axes[0, 1].legend(["Empleo adecuado", "Subempleo"], frameon=False)
axes[0, 1].set_title("Calidad del empleo de graduados")
axes[0, 1].set_ylabel("% de la PEA graduada")

axes[1, 0].bar(kpi["anio"], kpi["tasa_sobrecalificacion"], color="#2E4057")
axes[1, 0].set_title("Tasa de sobrecalificación (CIUO-08 grupos 4-9)")
axes[1, 0].set_ylabel("% de graduados ocupados")

comp = gold.comparativo_nivel.query("anio == @ultimo").nlargest(8, "ingreso_laboral_medio")
axes[1, 1].barh(comp["nivel_instruccion"], comp["ingreso_laboral_medio"], color="#6C8EAD")
axes[1, 1].set_title(f"Ingreso laboral medio por nivel de instrucción ({ultimo})")
axes[1, 1].set_xlabel("USD/mes")

fig.suptitle("ENEMDU — Empleabilidad de graduados universitarios (Gold)", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Brecha de género y dispersión territorial
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sexo = (gold.por_sexo_edad.query("confiable")
        .groupby(["anio", "sexo"])["tasa_desempleo"].mean().unstack())
sexo.plot(marker="o", ax=axes[0], color={"Hombre": "#2E4057", "Mujer": "#FF6B35"})
axes[0].set_title("Desempleo de graduados por sexo")
axes[0].set_ylabel("%")
axes[0].legend(frameon=False)

top = gold.por_provincia.query("anio == @ultimo and confiable").nlargest(12, "tasa_sobrecalificacion")
axes[1].barh(top["provincia"], top["tasa_sobrecalificacion"], color="#1B998B")
axes[1].invert_yaxis()
axes[1].set_title(f"Sobrecalificación por provincia ({ultimo})")
axes[1].set_xlabel("% de graduados ocupados")

fig.tight_layout()
plt.show()

### 5.4 Exportación para Power BI

In [ ]:
export_dir = paths.reports / "powerbi"
archivos = sorted(export_dir.glob("*.csv"))

print(f"📤 {len(archivos)} tablas exportadas en {export_dir}\n")
for f in archivos:
    n = sum(1 for _ in f.open(encoding="utf-8-sig")) - 1
    print(f"  {f.name:<40} {n:>6} filas   {f.stat().st_size/1e3:>7.1f} KB")

print("\nModelo sugerido en Power BI:")
print("  · Tabla de hechos : empleabilidad_graduados (anio × provincia × área × sexo × grupo_edad × nivel)")
print("  · Medidas DAX     : recalcular tasas como SUM(numerador)/SUM(denominador), NUNCA promediar tasas")
print("  · Filtro de calidad: usar la columna 'confiable' (n_muestral >= 30)")

---
## 6. Cierre y siguientes pasos

**Estado del pipeline**

| Capa | Artefacto | Contenido |
|---|---|---|
| 🥉 Bronze | `data/bronze/externas/enemdu/` | CSV + XLSX originales (inmutables) |
| 🥈 Silver | `data/silver/enemdu/enemdu_unificado.parquet` | 2021-2025 unificado, limpio y decodificado |
| 🥇 Gold | `data/gold/enemdu/*.parquet` | 7 tablas agregadas + CSV para Power BI |
| 📄 Reportes | `reports/schemas/*.json` | Esquemas, calidad de Silver y manifiesto de Gold |

**Ejecución reproducible fuera del notebook**
```bash
python scripts/run_enemdu_pipeline.py --all
python scripts/run_enemdu_pipeline.py --silver --gold --years 2024 2025
```

**Siguientes pasos sugeridos**
1. **Validar los mapeos con `mapeo_educacion`/`mapeo_condact` en `fallback`**: si algún año cae en fallback, revisar el diccionario y ajustar `EDU_PATTERNS`/`CONDACT_PATTERNS` en `enemdu_mappings.py`.
2. **Refinar la sobrecalificación**: complementar el criterio normativo CIUO-08 con el criterio estadístico (modal por ocupación) y comparar resultados.
3. **Cruzar con datos internos** (graduados UCUENCA) para pasar de una línea base nacional a una comparación institucional.
4. **Errores estándar**: incorporar el diseño muestral complejo (conglomerado/estrato) si se van a publicar intervalos de confianza.